# H3: level offsets of ODYSSEY periods A and B, checked with CERRA-Land

H3 (#20), second pull request (comment of 13/11/2026 on #79; H3b, #82, and H4, #23, merged into H3). ODYSSEY is
three products in a row (`docs/method.md`, *Radars over time*): A 4/2013–11/2015, B 12/2015–7/2018, C 8/2018–6/2024;
D is the target, CIRRUS 7/2024–17/11/2025. Period C is in notebook 07, section 12; here the same procedure for A
and B, with ERA5-Land as the bridge, and CERRA-Land as the independent check of ERA5-Land at 12/2015 and 8/2018.

1. **Data:** per-cell monthly sums of radar and ERA5-Land, as notebook 07 section 12, from 4/2013; Belgian-area
   daily sums on 06–06 UTC days of radar, ERA5-Land and CERRA-Land.
2. **Offsets of A, B (and C) against D** with ERA5-Land.
3. **CERRA-Land check** of the steps at 12/2015 and 8/2018, and offsets with CERRA-Land as bridge.
4. **Seasons, rings of distance from the radar, continuity** at 12/2015, 8/2018 and 7/2024 with years left out.
5. **Figure.**

Results in `docs/method.md`, *Level offsets of periods A and B (second H3 pull request)*.

In [ ]:
from opera_ds.paths import DATA_DIR, RESULTS_DIR

import calendar
import datetime as dt
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from scipy import stats
from scipy.stats import f_oneway, mannwhitneyu, ttest_ind

OUT = RESULTS_DIR / "h3_level"                      # hourly sums on the ERA5-Land grid, notebook 07 sections 4 and 9
FIG = Path.cwd().parent / "docs" / "figures" / "h3-level"
FULL_FRAMES = {"odyssey": 4, "cirrus": 12}
D_END = np.datetime64("2025-11-18T00")             # target D: CIRRUS to 17/11/2025
PERIODS = {"A": ((2013, 4), (2015, 11)), "B": ((2015, 12), (2018, 7)), "C": ((2018, 8), (2024, 6)),
           "D": ((2024, 7), (2025, 11))}

era = xr.open_zarr(DATA_DIR / "era5land_bel.zarr", consolidated=False)
expver = era["expver"].load()


def full_cells(product):
    z = np.load(OUT / f"era5grid_{product}_2024-07.npz")
    return {tuple(c) for c in z["cells"][z["npix"] >= 0.8 * z["npix"].max()]}


COMMON = sorted(full_cells("odyssey") & full_cells("cirrus"))     # the 186 cells of notebook 07
CI, CJ = np.array([c[0] for c in COMMON]), np.array([c[1] for c in COMMON])
print(len(COMMON), "cells")

## 1. Data

- **Monthly, per cell:** complete radar hours and final ERA5-Land, ODYSSEY cleaned as in H2 (notebook 07, section
  9), CIRRUS as it is; months with less than half of their hours are left out (11/2025: of its 17 days).
- **Daily, 06–06 UTC, Belgian area:** the hours where all 186 cells are finite in both are averaged over the
  cells; the day labelled *d* 06 UTC takes the hours ending in (*d*−1 06, *d* 06], as CERRA-Land (notebook 08);
  only days with all 24 hours. CERRA-Land is averaged over its cells whose centre lies in the 0.1° box of one of
  the 186 cells. So ODYSSEY is not recomputed on the CERRA-Land grid: the check uses the same area.

In [ ]:
MONTHS, H, RM, EM = {}, [], [], []
for y in range(2013, 2026):
    for m in range(1, 13):
        if not (2013, 4) <= (y, m) <= (2025, 11):
            continue
        product = "odyssey" if (y, m) < (2024, 7) else "cirrus"
        ym = f"{y}-{m:02d}"
        z = np.load(OUT / (f"era5grid_odyssey_clean_{ym}.npz" if product == "odyssey" else f"era5grid_cirrus_{ym}.npz"))
        pos = {tuple(c): k for k, c in enumerate(z["cells"])}
        sel = np.array([pos[c] for c in COMMON])
        he = np.array(z["hour_end"], dtype="datetime64[ns]")
        ok = (z["frames"] == FULL_FRAMES[product]) & (he <= era["valid_time"].values[-1]) & (he <= D_END)
        final = expver.sel(valid_time=he[ok]).values == 1
        hours = he[ok][final]
        r = (z["rain_clean"] if product == "odyssey" else z["rain"])[ok][final][:, sel]
        e = era["tp"].sel(valid_time=hours).values[:, CI, CJ]
        good = np.isfinite(r) & np.isfinite(e)
        H.append(hours[good.all(1)]); RM.append(r[good.all(1)].mean(1)); EM.append(e[good.all(1)].mean(1))
        nmax = (17 if ym == "2025-11" else calendar.monthrange(y, m)[1]) * 24
        if hours.size >= 0.5 * nmax and final.mean() >= 0.5:
            MONTHS[(y, m)] = dict(radar=np.where(good, r, 0).sum(0), era5=np.where(good, e, 0).sum(0))
KEYS = {p: [k for k in MONTHS if a <= k <= b] for p, (a, b) in PERIODS.items()}

H, RM, EM = np.concatenate(H).astype("datetime64[h]"), np.concatenate(RM), np.concatenate(EM)
lab = (H - np.timedelta64(7, "h")).astype("datetime64[D]").astype("datetime64[h]") + np.timedelta64(30, "h")
days, inv, cnt = np.unique(lab, return_inverse=True, return_counts=True)
rd, ed = np.bincount(inv, RM)[cnt == 24], np.bincount(inv, EM)[cnt == 24]
days = days[cnt == 24]
cer = xr.open_zarr(DATA_DIR / "cerraland_bel.zarr", consolidated=False)
inside = np.zeros(cer.latitude.shape, bool)
for la, lo in zip(era["latitude"].values[CI], era["longitude"].values[CJ]):
    inside |= (np.abs(cer.latitude.values - la) <= 0.05) & (np.abs(cer.longitude.values - lo) <= 0.05)
cv = cer.valid_time.values.astype("datetime64[h]")
keep = np.isin(days, cv)
days, rd, ed = days[keep], rd[keep], ed[keep]
cd = np.nanmean(cer["tp"].values[np.isin(cv, days)][:, inside], 1)
print(f"{len(MONTHS)} months {min(MONTHS)} → {max(MONTHS)}; {', '.join(f'{p} {len(v)}' for p, v in KEYS.items())}")
print(f"{days.size} complete 06–06 days; {inside.sum()} CERRA-Land cells; day check, correlation ERA5-Land /"
      f" CERRA-Land same day {np.corrcoef(ed, cd)[0, 1]:.3f}, one day apart {np.corrcoef(ed[1:], cd[:-1])[0, 1]:.3f}")

## 2. Offsets of A, B and C against D, ERA5-Land

As notebook 07, section 12: log10 of radar / ERA5-Land per month; for each calendar month, the mean over D minus
the mean over the period; the offset is the mean of the 12 differences, with a 95% interval (t, 11 degrees of
freedom); rain × k is +16·log10 k dB. Checks without the artefact months of B (spring 2018) and the summer echoes
near the radars of A (summers 2014 and 2015; `docs/method.md`, *Radars over time*).

In [ ]:
def logr(keys, cells=slice(None)):
    return np.array([np.log10(MONTHS[k]["radar"][cells].sum() / MONTHS[k]["era5"][cells].sum()) for k in keys])


def offset(c_keys, d_keys, **kw):
    lc, ld = logr(c_keys, **kw), logr(d_keys, **kw)
    mc, md = np.array([k[1] for k in c_keys]), np.array([k[1] for k in d_keys])
    d = np.array([ld[md == m].mean() - lc[mc == m].mean() for m in range(1, 13)])
    half = stats.t.ppf(0.975, 11) * d.std(ddof=1) / np.sqrt(12)
    return dict(off=d.mean(), lo=d.mean() - half, hi=d.mean() + half, by_month=d, unbalanced=ld.mean() - lc.mean())


def show(name, o):
    print(f"{name:46s} {o['off']:+.3f} (×{10 ** o['off']:.2f}, {16 * o['off']:+.2f} dB), 95% ×{10 ** o['lo']:.2f}–"
          f"{10 ** o['hi']:.2f} ({16 * o['lo']:+.2f} to {16 * o['hi']:+.2f} dB); unbalanced {o['unbalanced']:+.3f}")


OFF = {p: offset(KEYS[p], KEYS["D"]) for p in "ABC"}
for p in "ABC":
    show(f"{p} against D", OFF[p])
show("B without 3–6/2018", offset([k for k in KEYS["B"] if not (2018, 3) <= k <= (2018, 6)], KEYS["D"]))
show("A without the summers 2014 and 2015", offset([k for k in KEYS["A"] if not (k[1] in (6, 7, 8) and k[0] in (2014, 2015))], KEYS["D"]))

## 3. CERRA-Land: the steps at 12/2015 and 8/2018

Monthly sums of the complete 06–06 days (all 152 months have at least 15). The step of `opera_ds.homog.step`
(12 and 24 months each side, t-test and Wilcoxon–Mann–Whitney) on radar / ERA5-Land, radar / CERRA-Land and
ERA5-Land / CERRA-Land: if the radar steps are the same with both bridges and the ratio of the bridges has no
step, the step is in the radar. Then the offsets of A, B and C against D with each bridge on the same days; the
rule of the comment of 13/11 on #79 (VMM against ERA5-Land): if the intervals overlap, ERA5-Land carries the
period, else the difference is an uncertainty to state at G1.

In [ ]:
from opera_ds.homog import step

ym = days.astype("datetime64[M]")
MON = {}
for m in np.unique(ym):
    k = ym == m
    if k.sum() >= 15:
        MON[(int(str(m)[:4]), int(str(m)[5:7]))] = dict(radar=rd[k].sum(), era5=ed[k].sum(), cerra=cd[k].sum())
print(len(MON), "months with at least 15 complete days")
SER = {"radar / ERA5-Land": ("radar", "era5"), "radar / CERRA-Land": ("radar", "cerra"),
       "ERA5-Land / CERRA-Land": ("era5", "cerra")}
for at in ((2015, 12), (2018, 8), (2024, 7)):
    for n in (12, 24):
        print(f"step at {at[1]}/{at[0]}, {n} months each side")
        for name, (a, b) in SER.items():
            s = step({k: v[a] / v[b] for k, v in MON.items()}, at, n)
            print(f"   {name:24s} {s['step']:+.3f} (×{10 ** s['step']:.2f}), t-test p {s['t_p']:.3f},"
                  f" Mann–Whitney p {s['mw_p']:.3f}, n {s['n_before']}/{s['n_after']}")


def offset_daily(p, den):
    lr = {k: np.log10(v["radar"] / v[den]) for k, v in MON.items()}
    a, b = PERIODS[p]
    d = np.array([np.mean([v for k, v in lr.items() if k[1] == m and PERIODS["D"][0] <= k]) -
                  np.mean([v for k, v in lr.items() if k[1] == m and a <= k <= b]) for m in range(1, 13)])
    half = stats.t.ppf(0.975, 11) * d.std(ddof=1) / np.sqrt(12)
    return d.mean(), d.mean() - half, d.mean() + half


print("\noffsets against D on the same 06–06 days")
for p in "ABC":
    (e, elo, ehi), (c, clo, chi) = offset_daily(p, "era5"), offset_daily(p, "cerra")
    print(f"   {p}: ERA5-Land {e:+.3f} (×{10 ** e:.2f}, ×{10 ** elo:.2f}–{10 ** ehi:.2f}), CERRA-Land {c:+.3f}"
          f" (×{10 ** c:.2f}, ×{10 ** clo:.2f}–{10 ** chi:.2f}); difference {c - e:+.3f};"
          f" intervals overlap: {max(elo, clo) <= min(ehi, chi)}")

## 4. Seasons, rings, continuity

- **Seasons:** as notebook 07, section 12; A and B have only 2–3 years of each season.
- **Rings:** distance of each cell from the nearest of the radars of notebook 07, section 11; the offset of each
  ring, and the local 12-month step of radar / ERA5-Land at 12/2015 and 8/2018 in each ring.
- **Continuity:** each period split into 12-month blocks aligned to its end (the first block may be shorter);
  each block is left out in turn, the offset refitted on the rest of its period and applied to it. The yardstick
  is the standard deviation of the corrected means of the full blocks of A, B and C, with its 95% interval
  (chi-square). The jump at a boundary is the corrected first 12 months after it (offset fitted without them;
  for D, CIRRUS as it is) minus the corrected last 12 months before it (offset fitted without them).

In [ ]:
SEASONS = {"DJF": (12, 1, 2), "MAM": (3, 4, 5), "JJA": (6, 7, 8), "SON": (9, 10, 11)}
for p in "AB":
    groups = [OFF[p]["by_month"][[m - 1 for m in ms]] for ms in SEASONS.values()]
    print(f"{p}: one-way ANOVA of the 12 monthly differences by season, p {f_oneway(*groups).pvalue:.2f}")
    for (s, ms), g in zip(SEASONS.items(), groups):
        ck = [k for k in KEYS[p] if k[1] in ms]
        lc, sy = logr(ck), np.array([k[0] + (k[1] == 12) for k in ck])
        yrs = [lc[sy == y].mean() for y in np.unique(sy) if (sy == y).sum() == 3]
        sd = np.std(yrs, ddof=1)
        dev = g.mean() - OFF[p]["off"]
        print(f"   {s}: offset {g.mean():+.3f} (×{10 ** g.mean():.2f}), minus the single offset {dev:+.3f};"
              f" year-to-year sd {sd:.3f} ({len(yrs)} years) → {'exceeds' if abs(dev) > sd else 'within'}")

RADARS_NEAR = ["bejab", "behel", "bewid", "bezav", "nldhl", "nlhrw", "frabb", "frave", "deess"]
DB = {r["odimcode"]: (float(r["longitude"]), float(r["latitude"]))
      for r in json.load(open(DATA_DIR / "opera_radars" / "OPERA_RADARS_DB_20261110.json", encoding="utf-8"))
      if r.get("odimcode") in RADARS_NEAR}


def km(lon1, lat1, lon2, lat2):
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))


DIST = np.min([km(era["longitude"].values[CJ], era["latitude"].values[CI], lo, la) for lo, la in DB.values()], axis=0)
print(f"\nrings, cells {DIST.min():.0f}–{DIST.max():.0f} km from the nearest radar")
for name, a, b in (("0–50 km", 0, 50), ("50–100 km", 50, 100)):
    cells = np.where((DIST >= a) & (DIST < b))[0]
    for p in "ABC":
        show(f"{p}, {name}, {cells.size} cells", offset(KEYS[p], KEYS["D"], cells=cells))
    for at_name, before, after in (("12/2015", KEYS["A"][-12:], KEYS["B"][:12]), ("8/2018", KEYS["B"][-12:], KEYS["C"][:12])):
        print(f"   {name}: local 12-month step at {at_name} {logr(after, cells).mean() - logr(before, cells).mean():+.3f}")

In [ ]:
def blocks(p):
    ks, out = KEYS[p], []
    while ks:
        out.insert(0, ks[-12:])
        ks = ks[:-12]
    return out


def corrected(p, blk, exclude=()):
    o = offset([k for k in KEYS[p] if k not in blk and k not in exclude], KEYS["D"])["off"]
    blk = [k for k in blk if k not in exclude]
    return (logr(blk) + o).mean(), o


ROWS = []
for p in "ABC":
    for blk in blocks(p):
        m, o = corrected(p, blk)
        ROWS.append((p, blk, m))
        print(f"{p} {blk[0][1]}/{blk[0][0]}–{blk[-1][1]}/{blk[-1][0]} ({len(blk)} months): offset of the rest {o:+.3f},"
              f" corrected mean {m:+.3f}")
full = np.array([m for p, blk, m in ROWS if len(blk) == 12])
n, YARD = full.size, full.std(ddof=1)
Y_LO, Y_HI = (YARD * np.sqrt((n - 1) / stats.chi2.ppf(q, n - 1)) for q in (0.975, 0.025))
print(f"yardstick: {YARD:.3f}, 95% {Y_LO:.3f}–{Y_HI:.3f} ({n} blocks)")
D12 = logr([k for k in KEYS["D"] if k <= (2025, 6)]).mean()
JUMPS = {}
for p, q in (("A", "B"), ("B", "C"), ("C", "D")):
    before = corrected(p, KEYS[p][-12:])[0]
    after = D12 if q == "D" else corrected(q, KEYS[q][:12])[0]
    JUMPS[(p, q)] = (before, after)
    print(f"jump {p}→{q}: {after - before:+.3f} (×{10 ** (after - before):.2f}) → {'within' if abs(after - before) <= YARD else 'larger than'} the yardstick")
spring18 = [(2018, m) for m in (3, 4, 5, 6)]
b8 = corrected("B", KEYS["B"][-12:], exclude=spring18)[0]
print(f"jump B→C without 3–6/2018 in the fit and in the block: {JUMPS[('B', 'C')][1] - b8:+.3f}")

## 5. Figure

In [ ]:
keys = [k for p in "ABCD" for k in KEYS[p]]
t = np.array([dt.datetime(y, m, 15) for y, m in keys])
lr = logr(keys)
per = np.array([next(p for p in "ABCD" if PERIODS[p][0] <= k <= PERIODS[p][1]) for k in keys])
corr = lr + np.array([OFF[p]["off"] if p in OFF else 0.0 for p in per])
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=False, constrained_layout=True, height_ratios=[2, 1])
ax = axes[0]
ax.plot(t[per != "D"], 10 ** lr[per != "D"], ".-", color="0.7", lw=0.7, label="ODYSSEY cleaned")
for p, colour in (("A", "tab:green"), ("B", "tab:orange"), ("C", "tab:blue")):
    k = per == p
    ax.plot(t[k], 10 ** corr[k], ".-", color=colour, lw=0.9, label=f"{p} × {10 ** OFF[p]['off']:.2f}")
ax.plot(t[per == "D"], 10 ** lr[per == "D"], ".-", color="tab:red", lw=0.9, label="CIRRUS, target D")
for p, blk, m in ROWS:
    ax.hlines(10 ** m, dt.datetime(*blk[0], 1), dt.datetime(*blk[-1], 28), color="k", lw=2, alpha=0.35)
ax.set_yscale("log")
ax.set_ylim(1.0, 11)
ax.set_yticks([1, 1.5, 2, 3, 4, 6, 8, 10])
ax.set_yticklabels(["1", "1.5", "2", "3", "4", "6", "8", "10"])
ax.yaxis.set_minor_formatter(plt.NullFormatter())
ax.set_ylabel("radar / ERA5-Land\n(monthly totals)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.04), ncol=5, fontsize=8.5)
ax.set_title("Belgium, 186 ERA5-Land cells: offsets of the ODYSSEY periods against CIRRUS 7/2024–17/11/2025\n"
             "grey bars: corrected years, offset fitted without them")
ax = axes[1]
mk = sorted(MON)
ax.plot([dt.datetime(y, m, 15) for y, m in mk], [MON[k]["era5"] / MON[k]["cerra"] for k in mk], ".-", color="tab:purple",
        lw=0.8, label="ERA5-Land / CERRA-Land, 06–06 UTC days")
ax.set_ylabel("ERA5-Land /\nCERRA-Land")
ax.legend(loc="upper left", fontsize=8.5)
for a in axes:
    for d in (dt.datetime(2015, 12, 1), dt.datetime(2018, 8, 1), dt.datetime(2024, 7, 1)):
        a.axvline(d, color="0.4", ls="--", lw=0.8)
    a.grid(alpha=0.3)
axes[1].set_xlim(axes[0].get_xlim())
fig.savefig(FIG / "level-offsets-periods.png", dpi=110)
plt.show()

## Findings

See `docs/method.md`, *Level offsets of periods A and B (second H3 pull request)*.